R# Data Source Connectivity Check

Verifies network access to all Allen Brain Institute data sources from this Databricks cluster.

**Expected constraint:** Cluster firewall may block some external downloads.  
**Workaround:** Local download + upload to Workspace for any blocked endpoints.

In [0]:
import requests
import time
from dataclasses import dataclass, field


@dataclass
class ConnectivityResult:
    name: str
    url: str
    status: str = "PENDING"
    status_code: int | None = None
    response_size: int | None = None
    latency_ms: float | None = None
    error: str | None = None


results: list[ConnectivityResult] = []

## 1. Allen Brain Atlas API — Base Endpoint

In [0]:
def check_api_base() -> ConnectivityResult:
    url = "https://api.brain-map.org/api/v2/data/query.json?criteria=model::Atlas,rma::options[num_rows$eq1]"
    result = ConnectivityResult(name="Allen API Base", url=url)
    try:
        start = time.monotonic()
        resp = requests.get(url, timeout=30)
        result.latency_ms = round((time.monotonic() - start) * 1000, 1)
        result.status_code = resp.status_code
        result.response_size = len(resp.content)
        if resp.status_code == 200 and resp.json().get("success"):
            result.status = "PASS"
            print(f"PASS — API responded in {result.latency_ms} ms, {result.response_size} bytes")
        else:
            result.status = "FAIL"
            print(f"FAIL — HTTP {resp.status_code}, body: {resp.text[:200]}")
    except Exception as e:
        result.status = "BLOCKED"
        result.error = str(e)
        print(f"BLOCKED — {e}")
    return result


results.append(check_api_base())

BLOCKED — ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))


## 2. Nissl Reference Dataset Metadata (ID: 100960033)

In [0]:
def check_nissl_dataset() -> ConnectivityResult:
    url = (
        "https://api.brain-map.org/api/v2/data/query.json"
        "?criteria=model::SectionDataSet,rma::criteria,[id$eq100960033]"
    )
    result = ConnectivityResult(name="Nissl Dataset Metadata", url=url)
    try:
        start = time.monotonic()
        resp = requests.get(url, timeout=30)
        result.latency_ms = round((time.monotonic() - start) * 1000, 1)
        result.status_code = resp.status_code
        result.response_size = len(resp.content)
        data = resp.json()
        if resp.status_code == 200 and data.get("success") and len(data.get("msg", [])) > 0:
            dataset = data["msg"][0]
            result.status = "PASS"
            print(f"PASS — Dataset found: id={dataset['id']}, "
                  f"plane_of_section_id={dataset.get('plane_of_section_id')}, "
                  f"section_thickness={dataset.get('section_thickness')}")
        else:
            result.status = "FAIL"
            print(f"FAIL — HTTP {resp.status_code}, no dataset returned")
    except Exception as e:
        result.status = "BLOCKED"
        result.error = str(e)
        print(f"BLOCKED — {e}")
    return result


results.append(check_nissl_dataset())

BLOCKED — ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))


## 3. Section Image List Query

In [0]:
# Fetch one section image ID from the Nissl dataset for use in subsequent tests.
SECTION_IMAGE_ID: int | None = None


def check_section_image_list() -> ConnectivityResult:
    global SECTION_IMAGE_ID  # needed to pass ID to downstream cells
    url = (
        "https://api.brain-map.org/api/v2/data/query.json"
        "?criteria=model::SectionImage,rma::criteria,[data_set_id$eq100960033]"
        ",rma::options[num_rows$eq1][order$eq'id']"
    )
    result = ConnectivityResult(name="Section Image List", url=url)
    try:
        start = time.monotonic()
        resp = requests.get(url, timeout=30)
        result.latency_ms = round((time.monotonic() - start) * 1000, 1)
        result.status_code = resp.status_code
        result.response_size = len(resp.content)
        data = resp.json()
        if resp.status_code == 200 and data.get("success") and len(data.get("msg", [])) > 0:
            image_record = data["msg"][0]
            SECTION_IMAGE_ID = image_record["id"]
            result.status = "PASS"
            print(f"PASS — Found section image id={SECTION_IMAGE_ID}, "
                  f"width={image_record.get('width')}, height={image_record.get('height')}")
        else:
            result.status = "FAIL"
            print(f"FAIL — HTTP {resp.status_code}, no images returned")
    except Exception as e:
        result.status = "BLOCKED"
        result.error = str(e)
        print(f"BLOCKED — {e}")
    return result


results.append(check_section_image_list())

BLOCKED — ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))


## 4. Section Image Download (Thumbnail)

In [0]:
def check_image_download(section_image_id: int | None) -> ConnectivityResult:
    """Download a heavily downsampled thumbnail (~200 px wide) to test image serving."""
    if section_image_id is None:
        result = ConnectivityResult(name="Image Download", url="N/A")
        result.status = "SKIPPED"
        result.error = "No section_image_id from previous step"
        print("SKIPPED — No section image ID available (previous step failed)")
        return result

    url = f"https://api.brain-map.org/api/v2/section_image_download/{section_image_id}?downsample=8&quality=50"
    result = ConnectivityResult(name="Image Download", url=url)
    try:
        start = time.monotonic()
        resp = requests.get(url, timeout=60)
        result.latency_ms = round((time.monotonic() - start) * 1000, 1)
        result.status_code = resp.status_code
        result.response_size = len(resp.content)
        content_type = resp.headers.get("Content-Type", "")
        if resp.status_code == 200 and "image" in content_type:
            result.status = "PASS"
            print(f"PASS — Downloaded {result.response_size:,} bytes, "
                  f"content-type={content_type}, latency={result.latency_ms} ms")
        else:
            result.status = "FAIL"
            print(f"FAIL — HTTP {resp.status_code}, content-type={content_type}")
    except Exception as e:
        result.status = "BLOCKED"
        result.error = str(e)
        print(f"BLOCKED — {e}")
    return result


results.append(check_image_download(SECTION_IMAGE_ID))

SKIPPED — No section image ID available (previous step failed)


## 5. SVG Annotation Download

In [0]:
def check_svg_download(section_image_id: int | None) -> ConnectivityResult:
    """Download the SVG structure overlay for a section image."""
    if section_image_id is None:
        result = ConnectivityResult(name="SVG Annotation", url="N/A")
        result.status = "SKIPPED"
        result.error = "No section_image_id from previous step"
        print("SKIPPED — No section image ID available")
        return result

    url = f"https://api.brain-map.org/api/v2/svg_download/{section_image_id}"
    result = ConnectivityResult(name="SVG Annotation", url=url)
    try:
        start = time.monotonic()
        resp = requests.get(url, timeout=30)
        result.latency_ms = round((time.monotonic() - start) * 1000, 1)
        result.status_code = resp.status_code
        result.response_size = len(resp.content)
        content_type = resp.headers.get("Content-Type", "")
        if resp.status_code == 200 and ("svg" in content_type or resp.text.strip().startswith("<")):
            result.status = "PASS"
            print(f"PASS — SVG downloaded, {result.response_size:,} bytes, latency={result.latency_ms} ms")
        else:
            result.status = "FAIL"
            print(f"FAIL — HTTP {resp.status_code}, content-type={content_type}")
    except Exception as e:
        result.status = "BLOCKED"
        result.error = str(e)
        print(f"BLOCKED — {e}")
    return result


results.append(check_svg_download(SECTION_IMAGE_ID))

SKIPPED — No section image ID available


## 6. Structure Ontology (Graph ID 1 — Adult Mouse)

In [0]:
def check_structure_ontology() -> ConnectivityResult:
    url = "https://api.brain-map.org/api/v2/structure_graph_download/1.json"
    result = ConnectivityResult(name="Structure Ontology", url=url)
    try:
        start = time.monotonic()
        resp = requests.get(url, timeout=30)
        result.latency_ms = round((time.monotonic() - start) * 1000, 1)
        result.status_code = resp.status_code
        result.response_size = len(resp.content)
        if resp.status_code == 200 and resp.json().get("success"):
            structures = resp.json()["msg"]
            result.status = "PASS"
            print(f"PASS — Ontology downloaded, {result.response_size:,} bytes, "
                  f"{len(structures)} top-level nodes, latency={result.latency_ms} ms")
        else:
            result.status = "FAIL"
            print(f"FAIL — HTTP {resp.status_code}")
    except Exception as e:
        result.status = "BLOCKED"
        result.error = str(e)
        print(f"BLOCKED — {e}")
    return result


results.append(check_structure_ontology())

BLOCKED — ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))


## 7. CCFv3 Download Server (HEAD request — no large download)

In [0]:
def check_ccfv3_download() -> ConnectivityResult:
    """HEAD request to the 25 µm annotation volume (~85 MB). Checks reachability without downloading."""
    url = "https://download.alleninstitute.org/informatics-archive/current-release/mouse_ccf/annotation/ccf_2017/annotation_25.nrrd"
    result = ConnectivityResult(name="CCFv3 Download Server", url=url)
    try:
        start = time.monotonic()
        resp = requests.head(url, timeout=30, allow_redirects=True)
        result.latency_ms = round((time.monotonic() - start) * 1000, 1)
        result.status_code = resp.status_code
        content_length = resp.headers.get("Content-Length")
        if resp.status_code == 200:
            size_mb = int(content_length) / (1024 * 1024) if content_length else None
            result.status = "PASS"
            size_str = f", file size={size_mb:.1f} MB" if size_mb else ""
            print(f"PASS — Server reachable{size_str}, latency={result.latency_ms} ms")
        elif resp.status_code in (301, 302, 303, 307, 308):
            result.status = "PASS (redirect)"
            print(f"PASS (redirect) — HTTP {resp.status_code} → {resp.headers.get('Location', 'unknown')}")
        else:
            result.status = "FAIL"
            print(f"FAIL — HTTP {resp.status_code}")
    except Exception as e:
        result.status = "BLOCKED"
        result.error = str(e)
        print(f"BLOCKED — {e}")
    return result


results.append(check_ccfv3_download())

BLOCKED — ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))


## 8. Human Brain Atlas API

In [0]:
def check_human_atlas() -> ConnectivityResult:
    """Query human brain atlas products to verify API access for human data."""
    url = (
        "https://api.brain-map.org/api/v2/data/query.json"
        "?criteria=model::Product,rma::criteria,[abbreviation$eq'HBA']"
    )
    result = ConnectivityResult(name="Human Brain Atlas API", url=url)
    try:
        start = time.monotonic()
        resp = requests.get(url, timeout=30)
        result.latency_ms = round((time.monotonic() - start) * 1000, 1)
        result.status_code = resp.status_code
        result.response_size = len(resp.content)
        data = resp.json()
        if resp.status_code == 200 and data.get("success"):
            result.status = "PASS"
            products = data.get("msg", [])
            print(f"PASS — {len(products)} product(s) found, latency={result.latency_ms} ms")
            for p in products:
                print(f"       {p.get('abbreviation')}: {p.get('name')}")
        else:
            result.status = "FAIL"
            print(f"FAIL — HTTP {resp.status_code}")
    except Exception as e:
        result.status = "BLOCKED"
        result.error = str(e)
        print(f"BLOCKED — {e}")
    return result


results.append(check_human_atlas())

BLOCKED — ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))


## 9. PyPI — AllenSDK Package Availability

In [0]:
def check_pypi() -> ConnectivityResult:
    """Check if PyPI is reachable (needed for pip install allensdk)."""
    url = "https://pypi.org/pypi/allensdk/json"
    result = ConnectivityResult(name="PyPI (allensdk)", url=url)
    try:
        start = time.monotonic()
        resp = requests.get(url, timeout=30)
        result.latency_ms = round((time.monotonic() - start) * 1000, 1)
        result.status_code = resp.status_code
        result.response_size = len(resp.content)
        if resp.status_code == 200:
            data = resp.json()
            latest_version = data.get("info", {}).get("version", "unknown")
            result.status = "PASS"
            print(f"PASS — allensdk latest version: {latest_version}, latency={result.latency_ms} ms")
        else:
            result.status = "FAIL"
            print(f"FAIL — HTTP {resp.status_code}")
    except Exception as e:
        result.status = "BLOCKED"
        result.error = str(e)
        print(f"BLOCKED — {e}")
    return result


results.append(check_pypi())

PASS — allensdk latest version: 2.16.2, latency=24.1 ms


## 10. HuggingFace Hub — Model Access (for DINOv2 / SAM 2)

In [0]:
def check_huggingface() -> ConnectivityResult:
    """Check if HuggingFace model hub is reachable (needed to download models)."""
    url = "https://huggingface.co/api/models/facebook/dinov2-large"
    result = ConnectivityResult(name="HuggingFace Hub", url=url)
    try:
        start = time.monotonic()
        resp = requests.get(url, timeout=30)
        result.latency_ms = round((time.monotonic() - start) * 1000, 1)
        result.status_code = resp.status_code
        result.response_size = len(resp.content)
        if resp.status_code == 200:
            data = resp.json()
            result.status = "PASS"
            print(f"PASS — Model info retrieved: {data.get('modelId')}, "
                  f"downloads={data.get('downloads', 'N/A')}, latency={result.latency_ms} ms")
        else:
            result.status = "FAIL"
            print(f"FAIL — HTTP {resp.status_code}")
    except Exception as e:
        result.status = "BLOCKED"
        result.error = str(e)
        print(f"BLOCKED — {e}")
    return result


results.append(check_huggingface())

PASS — Model info retrieved: facebook/dinov2-large, downloads=890681, latency=45.9 ms


---

## Summary

In [ ]:
def print_summary(check_results: list[ConnectivityResult]) -> None:
    header = f"{'#':<3} {'Check':<28} {'Status':<16} {'HTTP':<6} {'Size':<12} {'Latency':<10} {'Error'}"
    separator = "-" * len(header)
    print("\n" + separator)
    print("CONNECTIVITY SUMMARY")
    print(separator)
    print(header)
    print(separator)

    passed = 0
    failed = 0
    blocked = 0

    for i, r in enumerate(check_results, 1):
        size_str = f"{r.response_size:,} B" if r.response_size else "-"
        latency_str = f"{r.latency_ms} ms" if r.latency_ms else "-"
        http_str = str(r.status_code) if r.status_code else "-"
        error_str = (r.error[:40] + "...") if r.error and len(r.error) > 40 else (r.error or "")
        print(f"{i:<3} {r.name:<28} {r.status:<16} {http_str:<6} {size_str:<12} {latency_str:<10} {error_str}")

        if r.status.startswith("PASS"):
            passed += 1
        elif r.status == "BLOCKED":
            blocked += 1
        else:
            failed += 1

    print(separator)
    print(f"\nTotal: {len(check_results)} checks — {passed} passed, {failed} failed, {blocked} blocked")

    if blocked > 0:
        print("\n⚠ BLOCKED endpoints detected. These need the Workspace upload workaround:")
        print("  1. Download the data locally (outside Databricks)")
        print("  2. Upload to Workspace: /Workspace/Users/noel.nosse@grainger.com/visual-model-ft/histology")
        print("  3. Read from Workspace path in notebooks")
        print("\nBlocked URLs:")
        for r in check_results:
            if r.status == "BLOCKED":
                print(f"  - {r.name}: {r.url}")
                print(f"    Error: {r.error}")
    elif passed == len(check_results):
        print("\nAll endpoints reachable — no firewall issues detected.")


print_summary(results)